In [ ]:
import re
import pandas as pd
import os
import glob
import matplotlib.pyplot as plt
import seaborn as sns

from IPython.core.interactiveshell import InteractiveShell
InteractiveShell.ast_node_interactivity = "all"

In [ ]:
def extract_stats(file_path):
    """Extract read statistics from a file."""
    with open(file_path, 'r') as f:
        content = f.read()
    
    # Extract numbers using regex
    total_umis = int(re.search(r'A total of (\d+) UMIs', content).group(1))
    genes_umis = int(re.search(r'(\d+) \([\d.]+%\) correspond to genes', content).group(1))
    locus_specific_umis = int(re.search(r'Locus-specific TEs: (\d+) UMIs', content).group(1))
    subfamily_umis = int(re.search(r'Subfamily TEs: (\d+)', content).group(1))
    
    # Get the sample name
    path_parts = file_path.split('/')
    dataset = path_parts[-3]  # simulated_mm_RA
    sample = path_parts[-2]      # young
    custom_name = f"{dataset}_{sample}"
    
    return {
        'dataset_sample': custom_name,
        'dataset': dataset,
        'sample': sample,
        'total_umis': total_umis,
        'genes_umis': genes_umis, 
        'locus_specific_umis': locus_specific_umis,
        'subfamily_umis': subfamily_umis
    }

def process_files(file_paths):
    """Process a list of file paths."""
    stats_list = []
    
    for file_path in file_paths:
        try:
            stats = extract_stats(file_path)
            stats_list.append(stats)
        except Exception as e:
            print(f"Error processing {file_path}: {e}")
    
    # Create DataFrame
    df = pd.DataFrame(stats_list)
    
    return df
    

In [ ]:
SoloTEpath="/mnt/volume_1p5T/results/SoloTEout"
dataset="simulated_mm_RA"
samples=["all","old","young"]

file_list = [f"{SoloTEpath}/{dataset}/{sample}/{sample}_SoloTE.stats" for sample in samples]
file_list

In [ ]:
SoloTEpath="/mnt/volume_1p5T/results/SoloTEout"
dataset="simulated_mm_RA"
samples=["all","old","young"]

file_list = [f"{SoloTEpath}/{dataset}/{sample}/{sample}_SoloTE.stats" for sample in samples]

df = process_files(file_list)
df
# df.to_csv("output_stats.csv", index=False)

# file_path = "/mnt/volume_1p5T/results/SoloTEout/simulated_mm_RA/young/young_SoloTE.stats"


In [ ]:
df['locus_specific_pct'] = df['locus_specific_umis'] / (df['locus_specific_umis'] + df['subfamily_umis']) * 100
df['subfamily_pct'] = 100 - df['locus_specific_pct']

melted_df = pd.melt(df,
    id_vars=['dataset_sample','dataset','sample'], 
    value_vars=['locus_specific_pct', 'subfamily_pct'],
    var_name='te_type', 
    value_name='percentage'
)
melted_df

In [ ]:

sns.set_context("poster")
sns.set_style("dark")
# Set up the figure
plt.figure(figsize=(5.3, 7))

df['sample'] = df['sample'].replace({
    'old': 'old TEs',
    'young': 'young TEs',
    'all': 'old+young TEs'
})

# Create stacked bar chart directly with matplotlib
ax = plt.subplot(111)
bar_width = 0.8

# Plot locus-specific bars
ax.bar(df['sample'], df['locus_specific_pct'], bar_width, 
       label='Unique', color='#76ABDF')

# Plot subfamily bars, stacked on top
ax.bar(df['sample'], df['subfamily_pct'], bar_width, 
       bottom=df['locus_specific_pct'], label='Multiple', color='#00487C')

# Customizing the plot
plt.title('', fontsize=16)
plt.xlabel('Simulations')
plt.ylabel("% of reads")
plt.xticks(rotation=45, ha='right')
plt.legend(title='Mapping')
plt.tight_layout()

plt.savefig("figures/percMultimapping_" + dataset + "_poster.png") 
plt.savefig("figures/percMultimapping_" + dataset + "_poster.pdf") 


In [ ]:

sns.set_context("talk")
sns.set_style("dark")
# Set up the figure
plt.figure(figsize=(4, 5.5))

df['sample'] = df['sample'].replace({
    'old': 'old TEs',
    'young': 'young TEs',
    'all': 'old+young TEs'
})
# order = ['old TEs', 'young TEs', 'old+young TEs']
# df['sample'] = pd.Categorical(df['sample'], categories=order, ordered=True)
# df = df.sort_values('sample')

# Create stacked bar chart directly with matplotlib
ax = plt.subplot(111)
bar_width = 0.8

# Plot locus-specific bars
ax.bar(df['sample'], df['locus_specific_pct'], bar_width, 
       label='Unique', color='#76ABDF')

# Plot subfamily bars, stacked on top
ax.bar(df['sample'], df['subfamily_pct'], bar_width, 
       bottom=df['locus_specific_pct'], label='Multiple', color='#00487C')

# Customizing the plot
plt.title('', fontsize=16)
plt.xlabel('Simulation')
plt.ylabel("% of reads")
plt.xticks(rotation=45, ha='right')
plt.legend(title='Mapping')
plt.tight_layout()

plt.savefig("figures/percMultimapping_" + dataset + "_talk.png") 
plt.savefig("figures/percMultimapping_" + dataset + "_talk|.pdf") 
